# Covariance shrinkage and the efficient frontier

**Sample covariance vs Ledoit-Wolf shrinkage, tested out of sample on 40 ETFs.**

This notebook builds Markowitz efficient frontiers, solves long-only minimum-variance and
maximum-Sharpe portfolios with turnover limits, and runs a monthly walk-forward backtest.
Each covariance estimator (sample, Ledoit-Wolf, OAS) drives an otherwise identical
strategy, so any difference in realized performance comes from the risk model alone.

| Cell | Purpose |
|---|---|
| 0 | Install dependencies (Colab) |
| 1 | Imports, logging, plotting theme |
| 2 | Configuration |
| 3 | ETF universe |
| 4 | Data collection (Yahoo Finance, cached, synthetic fallback) |
| 5 | Data cleaning and quality report |
| 6 | Feature engineering: returns, risk-free rate, rebalance calendar |
| 7 | Estimators: expected returns and covariance |
| 8 | Convex optimizer (cvxpy, DPP-compiled) |
| 9 | Efficient frontier and random portfolios |
| 10 | Walk-forward backtest engine |
| 11 | Performance metrics and Sharpe-difference test |
| 12 | Run the study |
| 13 | Visualizations |
| 14 | Lookback sensitivity (estimation error vs T/N) |
| 15 | Export dashboard and results |

## Cell 0 · Install dependencies
Colab ships with numpy, pandas, scikit-learn and plotly. cvxpy and yfinance need installing.

In [ ]:
%pip install -q cvxpy yfinance plotly scikit-learn pandas numpy

## Cell 1 · Imports, logging and plotting theme

In [ ]:
from __future__ import annotations

import logging
import math
import time
import warnings
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import cvxpy as cp
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from sklearn.covariance import OAS, LedoitWolf

warnings.filterwarnings("ignore", category=FutureWarning)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
log = logging.getLogger("shrinkage_frontier")

TRADING_DAYS = 252

# Colour roles. Estimator identity uses the first three categorical slots of a
# colour-blind-validated palette; benchmarks are neutral grey so they recede.
COLORS = {
    "ledoit_wolf": "#2a78d6",   # blue
    "sample": "#eb6834",        # orange
    "oas": "#1baf7a",           # aqua
    "equal_weight": "#8a8985",  # neutral grey
    "random": "#c9c8c2",
    "asset": "#52514e",
}
LABELS = {
    "sample": "Sample",
    "ledoit_wolf": "Ledoit-Wolf",
    "oas": "OAS",
    "equal_weight": "Equal weight",
    "min_variance": "Min variance",
    "max_sharpe": "Max Sharpe",
}
INK, INK_2, INK_3 = "#0b0b0b", "#52514e", "#8a8985"
SURFACE, GRID = "#fcfcfb", "#e6e5e0"
SEQUENTIAL_BLUE = [
    [0.0, "#f4f8fd"], [0.1, "#cde2fb"], [0.25, "#9ec5f4"], [0.4, "#6da7ec"],
    [0.55, "#3987e5"], [0.7, "#256abf"], [0.85, "#184f95"], [1.0, "#0d366b"],
]

pio.templates["desk"] = go.layout.Template(
    layout=dict(
        font=dict(family="Inter, Helvetica Neue, Arial, sans-serif", size=13, color=INK),
        title=dict(font=dict(size=18, color=INK), x=0.0, xanchor="left", y=0.97),
        paper_bgcolor=SURFACE,
        plot_bgcolor=SURFACE,
        colorway=[COLORS["ledoit_wolf"], COLORS["sample"], COLORS["oas"], COLORS["equal_weight"]],
        xaxis=dict(gridcolor=GRID, zerolinecolor=GRID, linecolor=GRID, tickfont=dict(color=INK_2),
                   title=dict(font=dict(color=INK_2))),
        yaxis=dict(gridcolor=GRID, zerolinecolor=GRID, linecolor=GRID, tickfont=dict(color=INK_2),
                   title=dict(font=dict(color=INK_2))),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0,
                    font=dict(color=INK_2), bgcolor="rgba(0,0,0,0)"),
        hoverlabel=dict(bgcolor="white", font=dict(color=INK), bordercolor=GRID),
        margin=dict(l=70, r=30, t=100, b=110),
        hovermode="closest",
    )
)
pio.templates.default = "plotly_white+desk"


def in_notebook() -> bool:
    """True when running inside Jupyter/Colab, where figures should render inline."""
    try:
        from IPython import get_ipython  # noqa: WPS433

        shell = get_ipython()
        return shell is not None and shell.__class__.__name__ in ("ZMQInteractiveShell", "Shell")
    except Exception:  # pragma: no cover - IPython missing
        return False


SHOW_FIGURES = in_notebook()


class DataDownloadError(RuntimeError):
    """Raised when market data cannot be retrieved and no fallback is allowed."""


class OptimizationError(RuntimeError):
    """Raised when every configured solver fails on a portfolio problem."""

## Cell 2 · Configuration
Every tunable lives here. Change one field and rerun from Cell 4.

In [ ]:
@dataclass
class Config:
    # Data
    start: str = "2008-01-01"
    end: Optional[str] = None                     # None = today
    risk_free_ticker: str = "^IRX"                # 13-week T-bill yield, % p.a.
    risk_free_fallback: float = 0.015             # used if ^IRX is unavailable
    benchmark: str = "SPY"
    max_missing_frac: float = 0.05                # drop tickers missing more than this
    ffill_limit: int = 5                          # max consecutive days to forward-fill
    allow_synthetic_fallback: bool = True         # simulate data if Yahoo is unreachable
    cache_max_age_days: int = 3

    # Estimation
    lookback_days: int = 252                      # rolling estimation window (T)
    estimators: Tuple[str, ...] = ("sample", "ledoit_wolf", "oas")
    shrink_correlation: bool = True               # shrink the correlation matrix, then rescale (see Cell 7)
    mu_method: str = "james_stein"                # "sample" | "james_stein"

    # Optimization
    objectives: Tuple[str, ...] = ("min_variance", "max_sharpe")
    w_max: float = 0.25                           # per-asset cap (long-only)
    turnover_cap: float = 0.50                    # L1 cap on |w_new - w_drifted| per rebalance
    tc_bps: float = 10.0                          # one-way transaction cost, basis points
    solvers: Tuple[str, ...] = ("CLARABEL", "OSQP", "SCS")

    # Frontier visuals
    frontier_points: int = 40
    frontier_w_max: float = 1.0                   # textbook frontier: long-only, no cap
    n_random_portfolios: int = 15_000
    frontier_split_frac: float = 0.55             # where in the sample to draw the frontier

    # Sensitivity study
    run_sensitivity: bool = True
    sensitivity_lookbacks: Tuple[int, ...] = (63, 126, 252, 504)
    sensitivity_w_max: float = 1.0                # loose constraints isolate the estimator effect
    sensitivity_classes: Tuple[str, ...] = ("US Equity", "Sector", "Intl Equity", "Real Estate")
    sensitivity_turnover_cap: float = 2.0         # 2.0 = no turnover limit

    # Misc
    seed: int = 42
    project_dir: Path = field(default_factory=lambda: Path.cwd())

    @property
    def data_dir(self) -> Path:
        return self.project_dir / "data"

    @property
    def output_dir(self) -> Path:
        return self.project_dir / "outputs"

    def validate(self, n_assets: Optional[int] = None) -> None:
        """Fail fast on settings that would make the optimizer infeasible or meaningless."""
        if not 0 < self.w_max <= 1:
            raise ValueError("w_max must be in (0, 1].")
        if n_assets is not None and self.w_max * n_assets < 1:
            raise ValueError(f"w_max={self.w_max} is infeasible for {n_assets} assets (needs w_max*N >= 1).")
        if not 0 < self.turnover_cap <= 2:
            raise ValueError("turnover_cap is an L1 distance between two long-only portfolios: use (0, 2].")
        if self.lookback_days < 21:
            raise ValueError("lookback_days below 21 leaves too few observations to estimate anything.")
        unknown = set(self.estimators) - {"sample", "ledoit_wolf", "oas"}
        if unknown:
            raise ValueError(f"Unknown estimators: {unknown}")
        unknown = set(self.objectives) - {"min_variance", "max_sharpe"}
        if unknown:
            raise ValueError(f"Unknown objectives: {unknown}")
        if self.mu_method not in ("sample", "james_stein"):
            raise ValueError("mu_method must be 'sample' or 'james_stein'.")


CFG = Config()
CFG.validate()
for d in (CFG.data_dir, CFG.output_dir, CFG.output_dir / "figures"):
    d.mkdir(parents=True, exist_ok=True)
np.random.seed(CFG.seed)
log.info("Project directory: %s", CFG.project_dir)

## Cell 3 · ETF universe
40 liquid ETFs across seven asset classes, all trading since at least January 2008. Mixing
equities, duration, credit and commodities gives the covariance matrix real structure: a
dominant equity factor, a negatively correlated bond block, and some near-duplicates
(SPY/DIA, IYR/VNQ, IEF/AGG) that make the sample estimate ill-conditioned.

In [ ]:
UNIVERSE: Dict[str, str] = {
    # US equity
    "SPY": "US Equity", "QQQ": "US Equity", "IWM": "US Equity", "DIA": "US Equity",
    "MDY": "US Equity", "VTV": "US Equity", "VUG": "US Equity",
    # US sectors
    "XLB": "Sector", "XLE": "Sector", "XLF": "Sector", "XLI": "Sector", "XLK": "Sector",
    "XLP": "Sector", "XLU": "Sector", "XLV": "Sector", "XLY": "Sector",
    # International equity
    "EFA": "Intl Equity", "EEM": "Intl Equity", "VGK": "Intl Equity", "EWJ": "Intl Equity",
    "EWZ": "Intl Equity", "FXI": "Intl Equity", "EWY": "Intl Equity", "EWT": "Intl Equity",
    "EWA": "Intl Equity", "EWC": "Intl Equity",
    # Real estate
    "IYR": "Real Estate", "VNQ": "Real Estate",
    # Government bonds
    "SHY": "Rates", "IEF": "Rates", "TLT": "Rates", "TIP": "Rates", "AGG": "Rates",
    # Credit
    "LQD": "Credit", "HYG": "Credit", "EMB": "Credit",
    # Commodities
    "GLD": "Commodities", "SLV": "Commodities", "DBC": "Commodities", "USO": "Commodities",
}
TICKERS: List[str] = list(UNIVERSE)
log.info("Universe: %d ETFs across %d asset classes", len(TICKERS), len(set(UNIVERSE.values())))

## Cell 4 · Data collection
Pipeline: **cache → Yahoo Finance (with retries) → synthetic fallback**.

* Prices are split- and dividend-adjusted (`auto_adjust=True`), so returns are total returns.
* Results are cached to CSV so reruns don't hit Yahoo again.
* If Yahoo is unreachable (rate limits, firewalled sandbox) and `allow_synthetic_fallback`
  is on, a factor model with stochastic volatility generates realistic prices so the rest of
  the pipeline still runs. Every downstream chart is labelled with the data source.

In [ ]:
def _read_cache(path: Path, max_age_days: int) -> Optional[pd.DataFrame]:
    if not path.exists():
        return None
    age_days = (time.time() - path.stat().st_mtime) / 86_400
    if age_days > max_age_days:
        log.info("Cache %s is %.1f days old; refreshing.", path.name, age_days)
        return None
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    log.info("Loaded %s from cache (%d rows x %d cols).", path.name, *df.shape)
    return df


def download_prices(tickers: Sequence[str], start: str, end: Optional[str],
                    retries: int = 3, pause: float = 2.0) -> pd.DataFrame:
    """Download adjusted close prices from Yahoo Finance with exponential backoff."""
    import yfinance as yf  # imported lazily so the module loads without network packages

    logging.getLogger("yfinance").setLevel(logging.CRITICAL)  # we log failures ourselves

    last_err: Optional[Exception] = None
    for attempt in range(1, retries + 1):
        try:
            raw = yf.download(list(tickers), start=start, end=end, auto_adjust=True,
                              progress=False, threads=True, group_by="column")
            if raw is None or raw.empty:
                raise DataDownloadError("Yahoo returned an empty frame.")
            px = raw["Close"] if isinstance(raw.columns, pd.MultiIndex) else raw[["Close"]]
            if isinstance(px, pd.Series):
                px = px.to_frame(tickers[0])
            px = px.dropna(how="all", axis=1)
            if px.empty:
                raise DataDownloadError("All tickers failed to download.")
            missing = sorted(set(tickers) - set(px.columns))
            if missing:
                log.warning("No data for: %s", ", ".join(missing))
            px.index = pd.to_datetime(px.index).tz_localize(None)
            return px.sort_index()
        except Exception as err:  # yfinance raises a wide range of exception types
            last_err = err
            wait = pause * 2 ** (attempt - 1)
            log.warning("Download attempt %d/%d failed (%s). Retrying in %.0fs.",
                        attempt, retries, err, wait)
            time.sleep(wait)
    raise DataDownloadError(f"Yahoo Finance download failed after {retries} attempts: {last_err}")


def simulate_prices(universe: Dict[str, str], start: str, end: Optional[str],
                    seed: int = 42) -> Tuple[pd.DataFrame, pd.Series]:
    """Generate synthetic ETF prices from a 5-factor model with stochastic volatility.

    Used only when Yahoo is unreachable. Factor loadings, drifts and vols are set per asset
    class so correlations look like the real thing (equity block, negative stock-bond
    correlation, commodity cluster). Innovations are Student-t (df=5) for fat tails, and a
    shared log-volatility AR(1) process produces crisis-like clusters.
    """
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range(start, end or pd.Timestamp.today().normalize())
    T, tickers = len(dates), list(universe)
    # Factors: equity, rates (duration), credit spread, commodities, EM/dollar
    factor_vol = np.array([0.16, 0.07, 0.05, 0.20, 0.10]) / np.sqrt(TRADING_DAYS)
    factor_drift = np.array([0.07, 0.01, 0.01, 0.00, 0.00]) / TRADING_DAYS
    profile = {  # (loadings, idio annual vol, extra annual drift)
        "US Equity": ([1.0, -0.1, 0.2, 0.1, 0.0], 0.06, 0.01),
        "Sector": ([1.0, -0.1, 0.2, 0.2, 0.0], 0.12, 0.00),
        "Intl Equity": ([0.9, -0.1, 0.2, 0.3, 0.9], 0.14, -0.02),
        "Real Estate": ([1.0, 0.4, 0.4, 0.0, 0.0], 0.10, 0.00),
        "Rates": ([-0.15, 1.0, 0.0, 0.0, 0.0], 0.02, 0.01),
        "Credit": ([0.25, 0.6, 1.0, 0.0, 0.2], 0.03, 0.01),
        "Commodities": ([0.3, 0.0, 0.1, 1.0, 0.3], 0.18, -0.01),
    }
    B = np.zeros((len(tickers), 5))
    idio, drift = np.zeros(len(tickers)), np.zeros(len(tickers))
    for i, t in enumerate(tickers):
        load, iv, dr = profile[universe[t]]
        B[i] = np.asarray(load) * rng.uniform(0.7, 1.3, 5)
        idio[i] = iv * rng.uniform(0.6, 1.4) / np.sqrt(TRADING_DAYS)
        drift[i] = dr / TRADING_DAYS
    if "SHY" in tickers:  # short duration: shrink the rates loading
        B[tickers.index("SHY")] *= 0.15
        idio[tickers.index("SHY")] *= 0.3

    # Shared stochastic volatility: log-vol AR(1), mean 0, persistence 0.985
    h = np.zeros(T)
    shocks = rng.normal(0, 0.11, T)
    for t in range(1, T):
        h[t] = 0.985 * h[t - 1] + shocks[t]
    vol_scale = np.exp(h - h.var() / 2)

    t_draws = rng.standard_t(5, size=(T, 5)) / np.sqrt(5 / 3)  # unit variance
    F = factor_drift + t_draws * factor_vol * vol_scale[:, None]
    eps = rng.standard_t(5, size=(T, len(tickers))) / np.sqrt(5 / 3) * idio * vol_scale[:, None]
    R = F @ B.T + eps + drift
    prices = pd.DataFrame(100 * np.cumprod(1 + R, axis=0), index=dates, columns=tickers)
    rf_annual = pd.Series(np.clip(0.015 + 0.01 * np.sin(np.arange(T) / 900), 0, None),
                          index=dates, name="rf")
    return prices, rf_annual


def load_market_data(cfg: Config, universe: Dict[str, str]) -> Tuple[pd.DataFrame, pd.Series, str]:
    """Return (prices, annual risk-free rate, source label). Cache -> Yahoo -> synthetic."""
    px_cache, rf_cache = cfg.data_dir / "prices.csv", cfg.data_dir / "risk_free.csv"
    tickers = list(universe)

    prices = _read_cache(px_cache, cfg.cache_max_age_days)
    rf = _read_cache(rf_cache, cfg.cache_max_age_days)
    if prices is not None and rf is not None and set(tickers) <= set(prices.columns):
        return prices[tickers], rf.iloc[:, 0], "Yahoo Finance (cached)"

    try:
        prices = download_prices(tickers, cfg.start, cfg.end)
        try:
            irx = download_prices([cfg.risk_free_ticker], cfg.start, cfg.end, retries=2)
            rf = (irx.iloc[:, 0] / 100.0).rename("rf")
        except DataDownloadError:
            log.warning("Risk-free series unavailable; using constant %.2f%%.", 100 * cfg.risk_free_fallback)
            rf = pd.Series(cfg.risk_free_fallback, index=prices.index, name="rf")
        prices.to_csv(px_cache)
        rf.to_frame().to_csv(rf_cache)
        log.info("Downloaded %d tickers, %d days (%s to %s).", prices.shape[1], prices.shape[0],
                 prices.index[0].date(), prices.index[-1].date())
        return prices, rf, "Yahoo Finance"
    except DataDownloadError as err:
        if not cfg.allow_synthetic_fallback:
            raise
        log.error("%s", err)
        log.warning("Falling back to SYNTHETIC data. Results illustrate the method, not markets.")
        prices, rf = simulate_prices(universe, cfg.start, cfg.end, cfg.seed)
        return prices, rf, "Synthetic (Yahoo unreachable)"


raw_prices, rf_annual_raw, DATA_SOURCE = load_market_data(CFG, UNIVERSE)
log.info("Data source: %s", DATA_SOURCE)

## Cell 5 · Data cleaning and quality report
Rules, applied in order:
1. Sort, de-duplicate dates, drop weekend rows and non-positive prices.
2. Drop tickers missing more than `max_missing_frac` of days in the study window.
3. Start the sample on the first date every surviving ticker has a price.
4. Forward-fill gaps of at most `ffill_limit` days (holidays on foreign listings), then drop
   any remaining incomplete rows.
5. Flag daily moves above 40% for review. Moves above 75% are almost always bad prints
   in adjusted ETF data, so they are zeroed.

In [ ]:
def clean_prices(prices: pd.DataFrame, cfg: Config) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Clean a wide price panel. Returns (clean prices, per-ticker quality report)."""
    px = prices.copy()
    px = px[~px.index.duplicated(keep="last")].sort_index()
    px = px[px.index.dayofweek < 5]
    px = px.mask(px <= 0)

    report = pd.DataFrame({
        "first_date": px.apply(lambda s: s.first_valid_index()),
        "last_date": px.apply(lambda s: s.last_valid_index()),
        "missing_frac": px.isna().mean(),
    })
    keep = report.index[report["missing_frac"] <= cfg.max_missing_frac]
    dropped = sorted(set(px.columns) - set(keep))
    if dropped:
        log.warning("Dropping %d tickers with sparse history: %s", len(dropped), ", ".join(dropped))
    px = px[keep]

    common_start = px.apply(lambda s: s.first_valid_index()).max()
    px = px.loc[common_start:]
    px = px.ffill(limit=cfg.ffill_limit).dropna(how="any")

    rets = px.pct_change().iloc[1:]
    report["kept"] = report.index.isin(keep)
    report["n_moves_gt_40pct"] = (rets.abs() > 0.40).sum().reindex(report.index).fillna(0).astype(int)
    report["ann_vol"] = (rets.std() * np.sqrt(TRADING_DAYS)).reindex(report.index)
    report["asset_class"] = report.index.map(UNIVERSE)

    if len(px) < cfg.lookback_days + 60:
        raise ValueError(f"Only {len(px)} clean rows; need at least lookback + 60.")
    log.info("Clean panel: %d tickers, %d days (%s to %s).", px.shape[1], len(px),
             px.index[0].date(), px.index[-1].date())
    return px, report


prices, quality_report = clean_prices(raw_prices, CFG)
quality_report.sort_values("ann_vol", ascending=False).head(10)

## Cell 6 · Feature engineering
* **Daily simple returns.** Portfolio returns aggregate linearly in simple returns, which is
  what the drift-and-rebalance accounting needs.
* **Daily risk-free rate.** The ^IRX yield is converted with `(1 + y)^(1/252) - 1` and
  aligned to trading days.
* **Rebalance calendar.** The last trading day of each month, starting once a full lookback
  window is available.

In [ ]:
def build_features(prices: pd.DataFrame, rf_annual: pd.Series, cfg: Config):
    returns = prices.pct_change().iloc[1:]
    bad = returns.abs() > 0.75
    if bad.values.any():
        log.warning("Zeroing %d daily moves above 75%% (likely bad prints).", int(bad.values.sum()))
        returns = returns.mask(bad, 0.0)

    rf_a = rf_annual.reindex(returns.index).ffill().bfill().fillna(cfg.risk_free_fallback)
    rf_daily = (1 + rf_a.clip(lower=0)) ** (1 / TRADING_DAYS) - 1

    month_ends = returns.index.to_series().groupby(returns.index.to_period("M")).last()
    positions = returns.index.get_indexer(month_ends.values)
    # Need a full lookback window before, and at least one holding day after.
    valid = (positions >= cfg.lookback_days - 1) & (positions < len(returns) - 1)
    return returns, rf_a, rf_daily, positions[valid]


returns, rf_annual, rf_daily, rebal_pos = build_features(prices, rf_annual_raw, CFG)
ASSETS = list(returns.columns)
N_ASSETS = len(ASSETS)
CFG.validate(N_ASSETS)
log.info("%d rebalances from %s to %s | N=%d, T=%d, T/N=%.1f",
         len(rebal_pos), returns.index[rebal_pos[0]].date(), returns.index[rebal_pos[-1]].date(),
         N_ASSETS, CFG.lookback_days, CFG.lookback_days / N_ASSETS)

## Cell 7 · Estimators
**Covariance.** With N assets and T observations, the sample covariance has N(N+1)/2 free
parameters (820 for N=40). Its extreme eigenvalues are biased: the largest come out too
large, the smallest too small. The optimizer then loads up on the directions that *look*
riskless, which are mostly estimation noise.

**Ledoit-Wolf** blends the sample matrix with a structured target:

$$\hat\Sigma_{LW} = \delta F + (1-\delta) S, \qquad F = \bar\lambda I$$

where $\bar\lambda$ is the average sample variance and $\delta$ is chosen analytically to
minimise expected Frobenius loss. That pulls eigenvalues toward their mean and lowers the
condition number. **OAS** (Oracle Approximating Shrinkage) uses the same target with an
intensity that converges faster under Gaussian data.

**Target choice.** scikit-learn's target is a *scaled identity*: one common variance for
every asset. In a multi-asset universe, SHY's variance is about 1/100th of EWZ's, so that
target drags bond variances up and equity variances down, and min-variance portfolios get
worse. With `shrink_correlation=True` (the default) the estimators run on standardised
returns and the result is rescaled by sample volatilities. Each asset keeps its own
variance and only the correlations shrink toward zero. That is the same as using a target
of diag(S). Set it to False to reproduce plain scikit-learn behaviour.

**Expected returns.** Sample means are noisier than covariances by an order of magnitude.
To keep the comparison about risk models, every strategy uses the same expected-return
vector: James-Stein shrinkage of sample means toward their cross-sectional average.

In [ ]:
def estimate_covariance(window: np.ndarray, method: str,
                        shrink_correlation: bool = True) -> Tuple[np.ndarray, float]:
    """Annualised covariance from a (T x N) daily return window. Returns (Sigma, shrinkage).

    With shrink_correlation=True, LW/OAS are fitted to standardised returns (a correlation
    matrix) and rescaled by sample volatilities, so the target is diag(S), not lambda*I.
    """
    if method == "sample":
        sigma, delta = np.cov(window, rowvar=False, ddof=1), 0.0
    elif method in ("ledoit_wolf", "oas"):
        model = LedoitWolf if method == "ledoit_wolf" else OAS
        if shrink_correlation:
            sd = window.std(axis=0, ddof=1)
            if np.any(sd <= 0):
                raise ValueError("Zero-variance asset in estimation window.")
            z = (window - window.mean(axis=0)) / sd
            est = model(assume_centered=True).fit(z)
            sigma = est.covariance_ * np.outer(sd, sd)
        else:
            est = model(assume_centered=False).fit(window)
            sigma = est.covariance_
        delta = float(est.shrinkage_)
    else:
        raise ValueError(f"Unknown covariance method: {method}")
    sigma = (sigma + sigma.T) / 2  # enforce exact symmetry
    return sigma * TRADING_DAYS, delta


def estimate_expected_returns(window: np.ndarray, method: str = "james_stein") -> Tuple[np.ndarray, float]:
    """Annualised expected returns. James-Stein shrinks toward the grand mean.

    delta = min(1, (N + 2) / ((N + 2) + T * d' S^-1 d)),  d = m - mean(m)
    S is a Ledoit-Wolf covariance so the inverse is stable even when T/N is small.
    """
    m = window.mean(axis=0)
    if method == "sample":
        return m * TRADING_DAYS, 0.0
    T, N = window.shape
    d = m - m.mean()
    S = LedoitWolf().fit(window).covariance_
    quad = float(T * d @ np.linalg.solve(S, d))
    delta = min(1.0, (N + 2) / ((N + 2) + quad))
    return ((1 - delta) * m + delta * m.mean()) * TRADING_DAYS, delta


def condition_number(sigma: np.ndarray) -> float:
    """lambda_max / lambda_min of the implied correlation matrix (inf if singular).

    Measured on correlations so the number reflects estimation noise, not the fact that
    SHY's variance is 100x smaller than EWZ's.
    """
    sd = np.sqrt(np.diag(sigma))
    eig = np.linalg.eigvalsh(sigma / np.outer(sd, sd))
    return float(eig[-1] / eig[0]) if eig[0] > 0 else float("inf")


def max_capped_return(mu: np.ndarray, w_max: float) -> Tuple[float, np.ndarray]:
    """Highest mu'w over {sum(w)=1, 0<=w<=w_max}. Greedy fill is the exact LP optimum."""
    w, remaining = np.zeros_like(mu, dtype=float), 1.0
    for i in np.argsort(mu)[::-1]:
        w[i] = min(w_max, remaining)
        remaining -= w[i]
        if remaining <= 1e-12:
            break
    return float(mu @ w), w


def cholesky_factor(sigma: np.ndarray) -> np.ndarray:
    """Upper factor U with U'U = Sigma, so w'Sigma w = ||U w||^2. Repairs near-singular input."""
    try:
        return np.linalg.cholesky(sigma + 1e-12 * np.eye(len(sigma))).T
    except np.linalg.LinAlgError:
        vals, vecs = np.linalg.eigh(sigma)
        vals = np.clip(vals, 1e-10 * max(vals.max(), 1e-12), None)
        log.debug("Covariance not PD; clipped %d eigenvalues.", int((vals <= 1e-10).sum()))
        return np.linalg.cholesky((vecs * vals) @ vecs.T).T


# Quick diagnostic on the latest window.
_w = returns.values[-CFG.lookback_days:]
for _m in CFG.estimators:
    _s, _d = estimate_covariance(_w, _m, CFG.shrink_correlation)
    log.info("%-12s condition number %10.1f  shrinkage %.3f", _m, condition_number(_s), _d)

## Cell 8 · Convex optimizer
Both problems are quadratic programs, written in cvxpy with **parameters** so each problem
is compiled once (DPP) and re-solved with new data at every rebalance. That is far faster
than rebuilding the problem 1,000+ times.

**Minimum variance**
$$\min_w \|Uw\|_2^2 \;\text{s.t.}\; \mathbf 1'w = 1,\; 0 \le w \le w_{max},\; \|w - w_{prev}\|_1 \le \tau$$

**Maximum Sharpe** is not convex as stated (a ratio), but the Cornuejols-Tutuncu change of
variables $y = k w,\ k > 0$ makes it a QP:
$$\min_{y,k} \|Uy\|_2^2 \;\text{s.t.}\; (\mu - r_f)'y = 1,\; \mathbf 1'y = k,\; 0 \le y \le w_{max} k,\; \|y - k\,w_{prev}\|_1 \le \tau k$$
and $w^* = y^*/k^*$. Every constraint is homogeneous in $(y, k)$, so caps and the turnover
limit carry over exactly. If no capped long-only portfolio has positive expected excess
return, the problem is infeasible and the strategy falls back to minimum variance that month.

**Turnover.** The constraint is measured against the *drifted* weights. If those violate
the per-asset cap after a big move, a hard turnover limit can be infeasible; the solver then
retries without the limit and records the event.

In [ ]:
@dataclass
class SolveInfo:
    status: str
    solver: str
    fallback: str = ""


class PortfolioOptimizer:
    """DPP-compiled long-only min-variance / max-Sharpe solver with turnover limits."""

    def __init__(self, n: int, w_max: float, solvers: Sequence[str]):
        self.n, self.w_max = n, w_max
        installed = set(cp.installed_solvers())
        self.solvers = [s for s in solvers if s in installed] or [None]
        self.U = cp.Parameter((n, n), name="U")
        self.mu_ex = cp.Parameter(n, name="mu_excess")
        self.w_prev = cp.Parameter(n, nonneg=True, name="w_prev")
        self.tau = cp.Parameter(nonneg=True, name="turnover_cap")

        self.w = cp.Variable(n, name="w")
        self._minvar = cp.Problem(
            cp.Minimize(cp.sum_squares(self.U @ self.w)),
            [cp.sum(self.w) == 1, self.w >= 0, self.w <= w_max,
             cp.norm1(self.w - self.w_prev) <= self.tau],
        )

        self.y = cp.Variable(n, name="y")
        self.k = cp.Variable(nonneg=True, name="k")
        self._maxsharpe = cp.Problem(
            cp.Minimize(cp.sum_squares(self.U @ self.y)),
            [self.mu_ex @ self.y == 1, cp.sum(self.y) == self.k, self.y >= 0,
             self.y <= w_max * self.k, cp.norm1(self.y - self.k * self.w_prev) <= self.tau * self.k],
        )
        assert self._minvar.is_dpp() and self._maxsharpe.is_dpp()

    def _run(self, problem: cp.Problem) -> Tuple[str, str]:
        last = "not_solved"
        for solver in self.solvers:
            try:
                problem.solve(solver=solver, warm_start=True)
                last = problem.status
                if problem.status in (cp.OPTIMAL, cp.OPTIMAL_INACCURATE):
                    return problem.status, str(solver)
            except cp.SolverError as err:
                last = f"solver_error:{err}"
        raise OptimizationError(last)

    @staticmethod
    def _clean(w: np.ndarray) -> np.ndarray:
        w = np.clip(np.asarray(w, dtype=float).ravel(), 0, None)
        w[w < 1e-6] = 0.0
        total = w.sum()
        if total <= 0 or not np.isfinite(total):
            raise OptimizationError("Solver returned a zero or non-finite weight vector.")
        return w / total

    def solve(self, sigma: np.ndarray, mu_excess: np.ndarray, objective: str,
              w_prev: Optional[np.ndarray], turnover_cap: float) -> Tuple[np.ndarray, SolveInfo]:
        """Solve one portfolio problem. w_prev=None means no turnover limit (initial trade)."""
        self.U.value = cholesky_factor(sigma)
        self.mu_ex.value = np.asarray(mu_excess, dtype=float)
        self.w_prev.value = np.full(self.n, 1 / self.n) if w_prev is None else np.clip(w_prev, 0, None)
        self.tau.value = 2.0 if w_prev is None else float(turnover_cap)  # 2.0 = never binding

        fallback = ""
        # Max Sharpe needs a feasible portfolio with positive excess return under the cap.
        if objective == "max_sharpe" and max_capped_return(np.asarray(mu_excess), self.w_max)[0] <= 1e-8:
            objective, fallback = "min_variance", "no_positive_excess_return"

        problem = self._maxsharpe if objective == "max_sharpe" else self._minvar
        try:
            status, solver = self._run(problem)
        except OptimizationError:
            self.tau.value = 2.0
            status, solver = self._run(problem)
            fallback = (fallback + "|" if fallback else "") + "turnover_relaxed"

        if objective == "max_sharpe":
            if self.k.value is None or self.k.value <= 1e-10:
                raise OptimizationError("Max-Sharpe returned k <= 0.")
            w = self._clean(self.y.value / self.k.value)
        else:
            w = self._clean(self.w.value)
        return w, SolveInfo(status, solver, fallback)


OPT = PortfolioOptimizer(N_ASSETS, CFG.w_max, CFG.solvers)
log.info("Optimizer ready. Solvers in priority order: %s", OPT.solvers)

## Cell 9 · Efficient frontier and random portfolios
The frontier traces minimum risk for each target return (long-only, no cap). Random
long-only portfolios come from a mix of Dirichlet(1) draws (diversified) and
Dirichlet(0.08) draws (concentrated), so the cloud fills the feasible region.

The second analysis asks the question the project is about: **how much of the risk the
frontier promises turns up out of sample?** Frontier portfolios are formed on one year of
data and held for the following year. The gap between ex-ante and realized volatility is
estimation error, made visible.

In [ ]:
class FrontierSolver:
    """Parametrised min-risk-for-target-return QP for tracing efficient frontiers."""

    def __init__(self, n: int, w_max: float, solvers: Sequence[str]):
        self.U = cp.Parameter((n, n))
        self.mu = cp.Parameter(n)
        self.target = cp.Parameter()
        self.w = cp.Variable(n)
        self.problem = cp.Problem(
            cp.Minimize(cp.sum_squares(self.U @ self.w)),
            [cp.sum(self.w) == 1, self.w >= 0, self.w <= w_max, self.mu @ self.w >= self.target],
        )
        installed = set(cp.installed_solvers())
        self.solvers = [s for s in solvers if s in installed] or [None]
        self.w_max = w_max

    def trace(self, sigma: np.ndarray, mu: np.ndarray, n_points: int) -> pd.DataFrame:
        self.U.value, self.mu.value = cholesky_factor(sigma), mu
        # Lower end: global min variance. Upper end: highest return achievable under the cap.
        self.target.value = -1e6
        w_gmv = self._solve()
        r_lo = float(mu @ w_gmv)
        r_hi, _ = max_capped_return(mu, self.w_max)
        rows = []
        for target in np.linspace(r_lo, r_lo + 0.999 * (r_hi - r_lo), n_points):
            self.target.value = target
            try:
                w = self._solve()
            except OptimizationError:
                continue
            rows.append({"target": target, "ret": float(mu @ w),
                         "vol": float(np.sqrt(w @ sigma @ w)), "weights": w})
        return pd.DataFrame(rows)

    def _solve(self) -> np.ndarray:
        for solver in self.solvers:
            try:
                self.problem.solve(solver=solver, warm_start=True)
                if self.problem.status in (cp.OPTIMAL, cp.OPTIMAL_INACCURATE):
                    return np.clip(self.w.value, 0, None) / np.clip(self.w.value, 0, None).sum()
            except cp.SolverError:
                continue
        raise OptimizationError(f"Frontier point failed: {self.problem.status}")


def random_portfolios(mu: np.ndarray, sigma: np.ndarray, n: int, rf: float, seed: int) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    k = len(mu)
    W = np.vstack([rng.dirichlet(np.full(k, 1.0), n // 2), rng.dirichlet(np.full(k, 0.08), n - n // 2)])
    ret = W @ mu
    vol = np.sqrt(np.einsum("ij,jk,ik->i", W, sigma, W))
    return pd.DataFrame({"ret": ret, "vol": vol, "sharpe": (ret - rf) / vol})


def frontier_study(returns: pd.DataFrame, rf_annual: pd.Series, cfg: Config) -> Dict[str, object]:
    """Estimated frontiers at one date, plus their realized risk over the following year."""
    R = returns.values
    split = int(cfg.frontier_split_frac * len(R))
    split = int(np.clip(split, cfg.lookback_days, len(R) - TRADING_DAYS))
    train, test = R[split - cfg.lookback_days:split], R[split:split + TRADING_DAYS]
    mu, _ = estimate_expected_returns(train, cfg.mu_method)
    rf = float(rf_annual.iloc[split - 1])
    fs = FrontierSolver(R.shape[1], cfg.frontier_w_max, cfg.solvers)
    test_cov = np.cov(test, rowvar=False) * TRADING_DAYS

    out: Dict[str, object] = {"date": returns.index[split - 1], "mu": mu, "rf": rf, "frontiers": {},
                              "test_end": returns.index[min(split + TRADING_DAYS, len(R)) - 1]}
    for est in cfg.estimators:
        sigma, _ = estimate_covariance(train, est, cfg.shrink_correlation)
        f = fs.trace(sigma, mu, cfg.frontier_points)
        f["realized_vol"] = [float(np.sqrt(w @ test_cov @ w)) for w in f["weights"]]
        f["realized_ret"] = [float(test.mean(axis=0) @ w * TRADING_DAYS) for w in f["weights"]]
        out["frontiers"][est] = {"sigma": sigma, "frontier": f}
    sample_sigma = out["frontiers"]["sample"]["sigma"] if "sample" in cfg.estimators else sigma
    out["random"] = random_portfolios(mu, sample_sigma, cfg.n_random_portfolios, rf, cfg.seed)
    out["asset_vol"] = np.sqrt(np.diag(sample_sigma))
    out["train"] = train
    return out


FRONTIER = frontier_study(returns, rf_annual, CFG)
for _est, _f in FRONTIER["frontiers"].items():
    _fr = _f["frontier"]
    log.info("%-12s frontier: mean ex-ante vol %.2f%% | mean realized vol %.2f%% | ratio %.2f",
             _est, 100 * _fr["vol"].mean(), 100 * _fr["realized_vol"].mean(),
             (_fr["realized_vol"] / _fr["vol"]).mean())

## Cell 10 · Walk-forward backtest engine
At each month-end rebalance date *t*:
1. Take the trailing `lookback_days` of returns ending at *t* (no look-ahead).
2. Estimate μ once, and Σ once per estimator.
3. For each (estimator, objective) pair, solve with turnover measured against that
   strategy's own drifted weights.
4. Charge `tc_bps` on traded notional on the first day of the holding period.
5. Hold to the next rebalance, letting weights drift with prices. Daily portfolio returns
   are computed exactly from the cumulative growth of each asset.

Equal weight (1/N, rebalanced monthly, same costs) runs in the same loop as a benchmark that
needs no estimation at all. DeMiguel, Garlappi and Uppal (2009) showed it is hard to beat.

In [ ]:
@dataclass
class StrategyResult:
    name: str
    estimator: str
    objective: str
    returns: pd.Series
    weights: pd.DataFrame
    diagnostics: pd.DataFrame


def _hold_period(w: np.ndarray, hold: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Daily returns of a buy-and-hold portfolio and its end-of-period drifted weights."""
    growth = np.cumprod(1 + hold, axis=0)  # (h, N)
    value = growth @ w                      # portfolio value path, starting at 1
    port = np.diff(np.concatenate([[1.0], value])) / np.concatenate([[1.0], value[:-1]])
    drifted = w * growth[-1] / value[-1]
    return port, drifted


def run_walk_forward(returns: pd.DataFrame, rf_annual: pd.Series, rebal_pos: np.ndarray,
                     cfg: Config, optimizer: PortfolioOptimizer, lookback: Optional[int] = None,
                     objectives: Optional[Sequence[str]] = None,
                     include_equal_weight: bool = True) -> Dict[str, StrategyResult]:
    lookback = lookback or cfg.lookback_days
    objectives = tuple(objectives or cfg.objectives)
    R, idx, n = returns.values, returns.index, returns.shape[1]
    rebal_pos = rebal_pos[rebal_pos >= lookback - 1]
    ends = np.append(rebal_pos[1:], len(R) - 1)

    keys = [(e, o) for e in cfg.estimators for o in objectives]
    if include_equal_weight:
        keys.append(("equal_weight", "equal_weight"))
    state = {k: {"w_prev": None, "rets": [], "dates": [], "weights": {}, "diag": []} for k in keys}
    cost_rate = cfg.tc_bps / 1e4
    t0 = time.time()

    for i, (pos, end) in enumerate(zip(rebal_pos, ends)):
        window = R[pos - lookback + 1:pos + 1]
        hold = R[pos + 1:end + 1]
        if len(hold) == 0:
            continue
        date = idx[pos]
        mu, _ = estimate_expected_returns(window, cfg.mu_method)
        mu_excess = mu - float(rf_annual.iloc[pos])
        sigmas = {e: estimate_covariance(window, e, cfg.shrink_correlation) for e in cfg.estimators}

        for key in keys:
            est, obj = key
            st = state[key]
            info, cond, delta = SolveInfo("fixed", "-"), np.nan, np.nan
            if est == "equal_weight":
                w = np.full(n, 1 / n)
                sigma = sigmas.get("ledoit_wolf", next(iter(sigmas.values())))[0]
            else:
                sigma, delta = sigmas[est]
                cond = condition_number(sigma)
                try:
                    w, info = optimizer.solve(sigma, mu_excess, obj, st["w_prev"], cfg.turnover_cap)
                except OptimizationError as err:
                    # Last resort: keep drifted weights (no trade) rather than crash the backtest.
                    log.warning("%s %s/%s failed (%s); holding previous weights.", date.date(), est, obj, err)
                    w = st["w_prev"] if st["w_prev"] is not None else np.full(n, 1 / n)
                    info = SolveInfo("failed_hold", "-", "solver_failed")

            prev = st["w_prev"] if st["w_prev"] is not None else np.zeros(n)
            turnover = float(np.abs(w - prev).sum())
            port, drifted = _hold_period(w, hold)
            port[0] -= turnover * cost_rate

            realized_vol = float(np.std(port, ddof=1) * np.sqrt(TRADING_DAYS)) if len(port) > 5 else np.nan
            st["rets"].append(port)
            st["dates"].append(idx[pos + 1:end + 1])
            st["weights"][date] = w
            st["diag"].append({
                "date": date, "turnover": turnover, "cond_number": cond, "shrinkage": delta,
                "pred_vol": float(np.sqrt(w @ sigma @ w)), "realized_vol": realized_vol,
                "n_holdings": int((w > 1e-4).sum()), "eff_n": float(1 / np.sum(w ** 2)),
                "max_weight": float(w.max()), "status": info.status, "fallback": info.fallback,
            })
            st["w_prev"] = drifted

        if (i + 1) % 24 == 0 or i == len(rebal_pos) - 1:
            log.info("  lookback %d | %3d/%d rebalances | %.1fs", lookback, i + 1, len(rebal_pos),
                     time.time() - t0)

    results: Dict[str, StrategyResult] = {}
    for (est, obj), st in state.items():
        name = "Equal weight" if est == "equal_weight" else f"{LABELS[est]} · {LABELS[obj]}"
        ser = pd.Series(np.concatenate(st["rets"]), index=np.concatenate(st["dates"]), name=name)
        wdf = pd.DataFrame(st["weights"], index=returns.columns).T
        results[name] = StrategyResult(name, est, obj, ser, wdf, pd.DataFrame(st["diag"]).set_index("date"))
    return results

## Cell 11 · Performance metrics
Return metrics use daily net-of-cost returns. Sharpe and Sortino use excess returns over
the daily T-bill rate. Portfolio-construction metrics come from the rebalance log.

| Metric | Why it matters here |
|---|---|
| Sharpe, Sortino, max drawdown, Calmar | Standard risk-adjusted performance |
| Realized volatility | Min-variance success is judged on this first |
| **Risk forecast ratio** = realized / predicted vol | Above 1 means the risk model is overconfident. The sample estimator should show the larger gap. |
| Annual turnover, effective N, max weight | Stability and concentration of the weights |
| Mean condition number | Numerical health of the covariance input |
| Memmel-corrected Jobson-Korkie test | Is the Sharpe difference vs sample covariance statistically real? |

In [ ]:
def performance_metrics(res: StrategyResult, rf_daily: pd.Series) -> Dict[str, float]:
    r = res.returns.dropna()
    ex = r - rf_daily.reindex(r.index).fillna(0)
    years = len(r) / TRADING_DAYS
    wealth = (1 + r).cumprod()
    dd = wealth / wealth.cummax() - 1
    cagr = wealth.iloc[-1] ** (1 / years) - 1
    vol = r.std() * np.sqrt(TRADING_DAYS)
    downside = ex[ex < 0].std() * np.sqrt(TRADING_DAYS)
    d = res.diagnostics
    return {
        "CAGR": cagr,
        "Volatility": vol,
        "Sharpe": ex.mean() / ex.std() * np.sqrt(TRADING_DAYS),
        "Sortino": ex.mean() * TRADING_DAYS / downside if downside > 0 else np.nan,
        "Max drawdown": dd.min(),
        "Calmar": cagr / abs(dd.min()) if dd.min() < 0 else np.nan,
        "Annual turnover": d["turnover"].iloc[1:].sum() / years,
        "Effective N": d["eff_n"].mean(),
        "Max weight": d["max_weight"].mean(),
        "Risk forecast ratio": (d["realized_vol"] / d["pred_vol"]).replace([np.inf, -np.inf], np.nan).median(),
        "Condition number": d["cond_number"].median(),
        "Fallbacks": int((d["fallback"] != "").sum()),
    }


def sharpe_difference_test(a: pd.Series, b: pd.Series, rf_daily: pd.Series) -> Tuple[float, float]:
    """Jobson-Korkie test with Memmel (2003) correction. H0: SR_a == SR_b. Returns (z, p)."""
    df = pd.concat([a, b], axis=1, join="inner").dropna()
    ex = df.sub(rf_daily.reindex(df.index).fillna(0), axis=0)
    T = len(ex)
    sa, sb = (ex.mean() / ex.std()).values
    rho = ex.corr().iloc[0, 1]
    var = (2 - 2 * rho + 0.5 * (sa ** 2 + sb ** 2 - 2 * sa * sb * rho ** 2)) / T
    z = (sa - sb) / math.sqrt(var) if var > 0 else float("nan")
    p = math.erfc(abs(z) / math.sqrt(2))
    return z, p


def metrics_table(results: Dict[str, StrategyResult], rf_daily: pd.Series) -> pd.DataFrame:
    rows = {}
    for name, res in results.items():
        m = performance_metrics(res, rf_daily)
        m["Estimator"], m["Objective"] = LABELS.get(res.estimator), LABELS.get(res.objective)
        sample_key = f"{LABELS['sample']} · {LABELS.get(res.objective, '')}"
        if res.estimator not in ("sample", "equal_weight") and sample_key in results:
            m["JK z vs sample"], m["JK p vs sample"] = sharpe_difference_test(
                res.returns, results[sample_key].returns, rf_daily)
        rows[name] = m
    cols = ["Estimator", "Objective", "CAGR", "Volatility", "Sharpe", "Sortino", "Max drawdown",
            "Calmar", "Annual turnover", "Effective N", "Max weight", "Risk forecast ratio",
            "Condition number", "JK z vs sample", "JK p vs sample", "Fallbacks"]
    return pd.DataFrame(rows).T.reindex(columns=cols)


def format_metrics(df: pd.DataFrame) -> pd.DataFrame:
    """Human-readable copy of the metrics table."""
    out = df.copy()
    pct = ["CAGR", "Volatility", "Max drawdown", "Max weight"]
    for c in pct:
        out[c] = out[c].map(lambda v: f"{v:.2%}" if pd.notna(v) else "")
    for c in ["Sharpe", "Sortino", "Calmar", "Risk forecast ratio", "JK z vs sample"]:
        out[c] = out[c].map(lambda v: f"{v:.2f}" if pd.notna(v) else "")
    out["Annual turnover"] = out["Annual turnover"].map(lambda v: f"{v:.0%}")
    out["Effective N"] = out["Effective N"].map(lambda v: f"{v:.1f}")
    out["Condition number"] = out["Condition number"].map(lambda v: f"{v:,.0f}" if pd.notna(v) else "")
    out["JK p vs sample"] = out["JK p vs sample"].map(lambda v: f"{v:.3f}" if pd.notna(v) else "")
    return out

## Cell 12 · Run the study

In [ ]:
log.info("Running walk-forward backtest: %d estimators x %d objectives + equal weight.",
         len(CFG.estimators), len(CFG.objectives))
RESULTS = run_walk_forward(returns, rf_annual, rebal_pos, CFG, OPT)

# Buy-and-hold benchmark over the same dates, for reference only.
_bt_index = next(iter(RESULTS.values())).returns.index
BENCH = returns[CFG.benchmark].reindex(_bt_index) if CFG.benchmark in returns else None

METRICS = metrics_table(RESULTS, rf_daily)
if BENCH is not None:
    _bench_res = StrategyResult(f"{CFG.benchmark} buy & hold", "benchmark", "benchmark", BENCH,
                                pd.DataFrame(), pd.DataFrame(
                                    {"turnover": [0.0], "eff_n": [1.0], "max_weight": [1.0], "pred_vol": [np.nan],
                                     "realized_vol": [np.nan], "cond_number": [np.nan], "fallback": [""]}))
    _bm = performance_metrics(_bench_res, rf_daily)
    _bm.update({"Estimator": "-", "Objective": "Buy & hold"})
    METRICS.loc[_bench_res.name] = pd.Series(_bm)

METRICS_DISPLAY = format_metrics(METRICS)
METRICS_DISPLAY

## Cell 13 · Visualizations
1. Efficient frontier with random portfolios (in-sample)
2. Promised vs realized risk along the frontier (out of sample)
3. Eigenvalue spectrum: sample vs shrunk, with the Marchenko-Pastur noise edge
4. Condition number and shrinkage intensity through time
5. Weight heatmaps over time: sample vs Ledoit-Wolf
6. Cumulative return and drawdown comparison
7. Rolling 12-month Sharpe ratio
8. Risk forecast accuracy and turnover
9. Summary metrics table

In [ ]:
FIGURES: Dict[str, go.Figure] = {}
_source_note = f"Data: {DATA_SOURCE}. Long-only, cap {CFG.w_max:.0%}, turnover cap {CFG.turnover_cap:.0%}, " \
               f"{CFG.tc_bps:.0f} bps costs, {CFG.lookback_days}-day lookback, monthly rebalance."


def _add_note(fig: go.Figure, text: str = _source_note) -> go.Figure:
    fig.add_annotation(text=text, xref="paper", yref="paper", x=0, y=0, yshift=-62, showarrow=False,
                       xanchor="left", yanchor="top", font=dict(size=11, color=INK_3))
    return fig


def _style_subplot_titles(fig: go.Figure) -> go.Figure:
    """Left-align subplot titles in secondary ink and lift the legend clear of them."""
    for a in fig.layout.annotations:
        a.update(font=dict(size=13, color=INK_2))
    fig.update_layout(legend=dict(y=1.12), margin=dict(t=125))
    return fig


def _register(key: str, fig: go.Figure) -> go.Figure:
    FIGURES[key] = fig
    if SHOW_FIGURES:
        fig.show()
    return fig


def fig_frontier(study: Dict[str, object]) -> go.Figure:
    mu, rf, rnd = study["mu"], study["rf"], study["random"]
    fig = go.Figure()
    fig.add_trace(go.Scattergl(
        x=rnd["vol"], y=rnd["ret"], mode="markers", name="Random long-only portfolios",
        marker=dict(size=4, color=rnd["sharpe"], colorscale=SEQUENTIAL_BLUE, opacity=0.55,
                    colorbar=dict(title="Sharpe", thickness=12, len=0.6, y=0.45)),
        hovertemplate="Vol %{x:.1%}<br>Return %{y:.1%}<br>Sharpe %{marker.color:.2f}<extra></extra>",
    ))
    fig.add_trace(go.Scatter(
        x=study["asset_vol"], y=mu, mode="markers+text", text=ASSETS, name="Individual ETFs",
        textposition="top center", textfont=dict(size=9, color=INK_2),
        marker=dict(size=8, color="white", line=dict(color=COLORS["asset"], width=1.5)),
        hovertemplate="%{text}<br>Vol %{x:.1%}<br>Return %{y:.1%}<extra></extra>",
    ))
    for est, d in study["frontiers"].items():
        f = d["frontier"]
        fig.add_trace(go.Scatter(
            x=f["vol"], y=f["ret"], mode="lines", name=f"{LABELS[est]} frontier",
            line=dict(color=COLORS[est], width=2.5),
            hovertemplate=f"{LABELS[est]}<br>Vol %{{x:.1%}}<br>Return %{{y:.1%}}<extra></extra>",
        ))
        sharpe = (f["ret"] - rf) / f["vol"]
        gmv, tan = f.iloc[0], f.iloc[int(sharpe.argmax())]
        fig.add_trace(go.Scatter(
            x=[gmv["vol"], tan["vol"]], y=[gmv["ret"], tan["ret"]], mode="markers", showlegend=False,
            marker=dict(size=12, symbol=["diamond", "star"], color=COLORS[est], line=dict(color="white", width=2)),
            text=["Min variance", "Max Sharpe"],
            hovertemplate=f"{LABELS[est]} · %{{text}}<br>Vol %{{x:.1%}}<br>Return %{{y:.1%}}<extra></extra>",
        ))
    fig.update_layout(
        title=f"Efficient frontier with random portfolios · estimation window ending {study['date']:%b %Y}",
        xaxis=dict(title="Annualised volatility (ex-ante)", tickformat=".0%", rangemode="tozero"),
        yaxis=dict(title="Expected return (James-Stein, annualised)", tickformat=".0%"),
        height=640,
    )
    fig.add_annotation(text="◆ min variance   ★ max Sharpe", xref="paper", yref="paper", x=1, y=1.06,
                       showarrow=False, xanchor="right", font=dict(size=11, color=INK_2))
    return _add_note(fig, f"Data: {DATA_SOURCE}. Random portfolios and ETF points use the sample covariance.")


def fig_promised_vs_realized(study: Dict[str, object]) -> go.Figure:
    fig = go.Figure()
    for est, d in study["frontiers"].items():
        f = d["frontier"]
        fig.add_trace(go.Scatter(
            x=f["vol"], y=f["ret"], mode="lines", name=f"{LABELS[est]} · promised",
            line=dict(color=COLORS[est], width=2, dash="dash"), legendgroup=est,
            hovertemplate=f"{LABELS[est]} ex-ante<br>Vol %{{x:.1%}}<br>Exp. return %{{y:.1%}}<extra></extra>",
        ))
        fig.add_trace(go.Scatter(
            x=f["realized_vol"], y=f["ret"], mode="lines+markers", name=f"{LABELS[est]} · realized",
            line=dict(color=COLORS[est], width=2), marker=dict(size=6), legendgroup=est,
            hovertemplate=f"{LABELS[est]} realized<br>Vol %{{x:.1%}}<br>Exp. return %{{y:.1%}}<extra></extra>",
        ))
    fig.update_layout(
        title=f"Promised vs realized risk · formed {study['date']:%b %Y}, held to {study['test_end']:%b %Y}",
        xaxis=dict(title="Annualised volatility", tickformat=".0%"),
        yaxis=dict(title="Ex-ante expected return (same μ for all)", tickformat=".0%"),
        height=560,
    )
    return _add_note(fig, "Dashed: risk the optimizer expected. Solid: risk delivered over the next 252 days. "
                          "The horizontal gap is covariance estimation error.")


def fig_eigen_spectrum(study: Dict[str, object]) -> go.Figure:
    T, N = study["train"].shape
    fig = go.Figure()
    for est, d in study["frontiers"].items():
        sd = np.sqrt(np.diag(d["sigma"]))
        eig = np.sort(np.linalg.eigvalsh(d["sigma"] / np.outer(sd, sd)))[::-1]
        fig.add_trace(go.Scatter(
            x=np.arange(1, N + 1), y=eig, mode="lines+markers", name=LABELS[est],
            line=dict(color=COLORS[est], width=2), marker=dict(size=6),
            hovertemplate=f"{LABELS[est]}<br>Rank %{{x}}<br>λ = %{{y:.3f}}<extra></extra>",
        ))
    # Marchenko-Pastur bounds: the eigenvalue range of a pure-noise correlation matrix.
    lam_plus, lam_minus = (1 + np.sqrt(N / T)) ** 2, (1 - np.sqrt(N / T)) ** 2
    fig.add_hrect(y0=lam_minus, y1=lam_plus, fillcolor=GRID, opacity=0.45, line_width=0, layer="below")
    fig.add_hline(y=lam_plus, line=dict(color=INK_3, dash="dot", width=1.5),
                  annotation_text="Marchenko-Pastur noise band", annotation_position="top right",
                  annotation_font=dict(size=11, color=INK_2))
    fig.update_layout(title=f"Eigenvalue spectrum of the correlation matrix · N={N}, T={T}",
                      xaxis=dict(title="Eigenvalue rank"),
                      yaxis=dict(title="Eigenvalue (log scale)", type="log"),
                      height=500)
    return _add_note(fig, "Eigenvalues inside the grey band are indistinguishable from noise. Shrinkage lifts the "
                          "smallest ones, which the sample estimate pushes toward zero and the optimizer then over-weights.")


def fig_condition_over_time(results: Dict[str, StrategyResult], objective: str) -> go.Figure:
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08, row_heights=[0.62, 0.38],
                        subplot_titles=("Condition number of the correlation matrix (log scale)", "Shrinkage intensity δ"))
    for res in results.values():
        if res.objective != objective or res.estimator == "equal_weight":
            continue
        d = res.diagnostics
        fig.add_trace(go.Scatter(x=d.index, y=d["cond_number"], name=LABELS[res.estimator], mode="lines",
                                 line=dict(color=COLORS[res.estimator], width=2), legendgroup=res.estimator,
                                 hovertemplate="%{x|%b %Y}<br>κ = %{y:,.0f}<extra>" + LABELS[res.estimator] + "</extra>"),
                      row=1, col=1)
        if res.estimator != "sample":
            fig.add_trace(go.Scatter(x=d.index, y=d["shrinkage"], name=LABELS[res.estimator], mode="lines",
                                     showlegend=False, legendgroup=res.estimator,
                                     line=dict(color=COLORS[res.estimator], width=2),
                                     hovertemplate="%{x|%b %Y}<br>δ = %{y:.3f}<extra>" + LABELS[res.estimator] + "</extra>"),
                          row=2, col=1)
    fig.update_yaxes(type="log", row=1, col=1)
    fig.update_yaxes(tickformat=".2f", row=2, col=1)
    fig.update_layout(title="Covariance conditioning through time", height=640, hovermode="x unified")
    return _add_note(_style_subplot_titles(fig))


def fig_weight_heatmaps(results: Dict[str, StrategyResult], objective: str,
                        estimators: Sequence[str] = ("sample", "ledoit_wolf")) -> go.Figure:
    order = sorted(ASSETS, key=lambda t: (list(dict.fromkeys(UNIVERSE.values())).index(UNIVERSE[t]), t))
    picked = [r for r in results.values() if r.objective == objective and r.estimator in estimators]
    picked.sort(key=lambda r: list(estimators).index(r.estimator))
    fig = make_subplots(rows=len(picked), cols=1, shared_xaxes=True, vertical_spacing=0.06,
                        subplot_titles=[f"{LABELS[r.estimator]} covariance" for r in picked])
    for i, r in enumerate(picked, start=1):
        W = r.weights[order]
        fig.add_trace(go.Heatmap(
            z=W.T.values, x=W.index, y=order, zmin=0, zmax=CFG.w_max, colorscale=SEQUENTIAL_BLUE,
            colorbar=dict(title="Weight", tickformat=".0%", thickness=12, len=0.5) if i == 1 else None,
            showscale=(i == 1), xgap=0, ygap=1,
            hovertemplate="%{x|%b %Y}<br>%{y}: %{z:.1%}<extra>" + LABELS[r.estimator] + "</extra>",
        ), row=i, col=1)
        fig.update_yaxes(autorange="reversed", tickfont=dict(size=9), tickmode="array",
                         tickvals=order, ticktext=order, row=i, col=1)
    fig.update_layout(title=f"Portfolio weights over time · {LABELS[objective]}", height=120 + 520 * len(picked))
    _style_subplot_titles(fig)
    return _add_note(fig, "ETFs grouped by asset class. Vertical banding means the optimizer swaps holdings "
                          "month to month; smooth horizontal bands mean stable allocations.")


def _drawdown(r: pd.Series) -> pd.Series:
    w = (1 + r).cumprod()
    return w / w.cummax() - 1


def fig_cumulative(results: Dict[str, StrategyResult], objectives: Sequence[str]) -> go.Figure:
    titles = [f"{LABELS[o]}: growth of $1" for o in objectives] + [f"{LABELS[o]}: drawdown" for o in objectives]
    fig = make_subplots(rows=2, cols=len(objectives), shared_xaxes=True, row_heights=[0.68, 0.32],
                        vertical_spacing=0.07, horizontal_spacing=0.07, subplot_titles=titles)
    ew = next((r for r in results.values() if r.estimator == "equal_weight"), None)
    for c, obj in enumerate(objectives, start=1):
        series = [r for r in results.values() if r.objective == obj] + ([ew] if ew else [])
        for r in series:
            color = COLORS[r.estimator]
            dash = "dash" if r.estimator == "equal_weight" else "solid"
            label = LABELS[r.estimator]
            wealth = (1 + r.returns).cumprod()
            fig.add_trace(go.Scatter(x=wealth.index, y=wealth, mode="lines", name=label, legendgroup=label,
                                     showlegend=(c == 1), line=dict(color=color, width=2, dash=dash),
                                     hovertemplate="%{y:$.2f}<extra>" + label + "</extra>"), row=1, col=c)
            dd = _drawdown(r.returns)
            fig.add_trace(go.Scatter(x=dd.index, y=dd, mode="lines", name=label, legendgroup=label,
                                     showlegend=False, line=dict(color=color, width=1.5, dash=dash),
                                     hovertemplate="%{y:.1%}<extra>" + label + "</extra>"), row=2, col=c)
        fig.update_yaxes(type="log", tickprefix="$", row=1, col=c)
        fig.update_yaxes(tickformat=".0%", row=2, col=c)
    fig.update_layout(title="Out-of-sample cumulative return and drawdown (net of costs)", height=740,
                      hovermode="x unified")
    return _add_note(_style_subplot_titles(fig))


def fig_rolling_sharpe(results: Dict[str, StrategyResult], objectives: Sequence[str],
                       rf_daily: pd.Series, window: int = TRADING_DAYS) -> go.Figure:
    fig = make_subplots(rows=1, cols=len(objectives), shared_yaxes=True, horizontal_spacing=0.05,
                        subplot_titles=[LABELS[o] for o in objectives])
    for c, obj in enumerate(objectives, start=1):
        for r in results.values():
            if r.objective != obj:
                continue
            ex = r.returns - rf_daily.reindex(r.returns.index).fillna(0)
            rs = (ex.rolling(window).mean() / ex.rolling(window).std() * np.sqrt(TRADING_DAYS)).dropna()
            fig.add_trace(go.Scatter(x=rs.index, y=rs, mode="lines", name=LABELS[r.estimator],
                                     legendgroup=r.estimator, showlegend=(c == 1),
                                     line=dict(color=COLORS[r.estimator], width=1.8),
                                     hovertemplate="%{y:.2f}<extra>" + LABELS[r.estimator] + "</extra>"),
                          row=1, col=c)
        fig.add_hline(y=0, line=dict(color=INK_3, width=1), row=1, col=c)
    fig.update_layout(title="Rolling 12-month Sharpe ratio", height=500, hovermode="x unified")
    return _add_note(_style_subplot_titles(fig))


def fig_risk_and_turnover(metrics: pd.DataFrame) -> go.Figure:
    m = metrics[metrics["Objective"].isin([LABELS[o] for o in CFG.objectives])].copy()
    est_key = {v: k for k, v in LABELS.items()}
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
                        subplot_titles=("Risk forecast ratio (realized ÷ predicted vol)", "Annual turnover"))
    for est_label in m["Estimator"].unique():
        sub = m[m["Estimator"] == est_label]
        color = COLORS[est_key[est_label]]
        for col, metric, fmt in ((1, "Risk forecast ratio", ".2f"), (2, "Annual turnover", ".0%")):
            fig.add_trace(go.Bar(x=sub["Objective"], y=sub[metric].astype(float), name=est_label,
                                 marker=dict(color=color, line=dict(color=SURFACE, width=2)),
                                 legendgroup=est_label, showlegend=(col == 1),
                                 texttemplate=f"%{{y:{fmt}}}", textposition="outside",
                                 textfont=dict(color=INK_2, size=11),
                                 hovertemplate=f"{est_label}<br>%{{x}}: %{{y:{fmt}}}<extra></extra>"),
                          row=1, col=col)
    fig.add_hline(y=1.0, line=dict(color=INK_3, dash="dot"), row=1, col=1,
                  annotation_text="perfect forecast", annotation_font=dict(size=10, color=INK_2))
    fig.update_yaxes(tickformat=".0%", row=1, col=2)
    fig.update_layout(title="Risk model accuracy and trading cost", barmode="group", height=520,
                      bargap=0.25, bargroupgap=0.08)
    _style_subplot_titles(fig)
    return _add_note(fig, "Median across monthly holding periods. A ratio above 1 means the optimizer "
                          "under-estimated the risk of the portfolio it chose.")


def fig_metrics_table(display: pd.DataFrame) -> go.Figure:
    cols = ["CAGR", "Volatility", "Sharpe", "Sortino", "Max drawdown", "Annual turnover",
            "Effective N", "Risk forecast ratio", "JK p vs sample"]
    df = display[cols].reset_index().rename(columns={"index": "Strategy"})
    fig = go.Figure(go.Table(
        columnwidth=[2.4] + [1] * len(cols),
        header=dict(values=[f"<b>{c}</b>" for c in df.columns], fill_color="#f0efec",
                    font=dict(color=INK, size=12), align=["left"] + ["right"] * len(cols), height=34),
        cells=dict(values=[df[c] for c in df.columns], fill_color=SURFACE, font=dict(color=INK_2, size=12),
                   align=["left"] + ["right"] * len(cols), height=30, line_color=GRID),
    ))
    fig.update_layout(title="Out-of-sample performance summary", height=110 + 32 * len(df),
                      margin=dict(l=20, r=20, t=60, b=10))
    return fig


_hm_obj = "max_sharpe" if "max_sharpe" in CFG.objectives else CFG.objectives[0]
_register("frontier", fig_frontier(FRONTIER))
_register("promised_vs_realized", fig_promised_vs_realized(FRONTIER))
_register("eigen_spectrum", fig_eigen_spectrum(FRONTIER))
_register("conditioning", fig_condition_over_time(RESULTS, CFG.objectives[0]))
_register("weights_heatmap", fig_weight_heatmaps(RESULTS, _hm_obj))
_register("cumulative", fig_cumulative(RESULTS, CFG.objectives))
_register("rolling_sharpe", fig_rolling_sharpe(RESULTS, CFG.objectives, rf_daily))
_register("risk_turnover", fig_risk_and_turnover(METRICS))
_register("metrics_table", fig_metrics_table(METRICS_DISPLAY))
log.info("Built %d figures.", len(FIGURES))

## Cell 14 · Lookback sensitivity
Shrinkage should help most when T/N is small. This cell reruns the minimum-variance
backtest for several lookbacks and compares realized volatility, which is the quantity
min-variance is trying to minimise. Expect the gap between sample and Ledoit-Wolf to
shrink as the window lengthens.

Two changes from the main backtest isolate the estimator effect:
* **Equity and REIT sleeve only (28 ETFs).** In the full universe an uncapped min-variance
  portfolio puts about 95% in SHY, whatever the estimator. The equity block is highly
  correlated, which is where sample covariance breaks down.
* **No per-asset cap, no turnover limit.** Jagannathan and Ma (2003) showed that weight
  constraints act like covariance shrinkage, so caps narrow the gap between estimators.

In [ ]:
def lookback_sensitivity(cfg: Config) -> pd.DataFrame:
    rows = []
    sleeve = [t for t in ASSETS if UNIVERSE.get(t) in cfg.sensitivity_classes]
    if len(sleeve) < 5:
        raise ValueError("Sensitivity sleeve has fewer than 5 assets; check sensitivity_classes.")
    loose_cfg = replace(cfg, w_max=cfg.sensitivity_w_max, turnover_cap=cfg.sensitivity_turnover_cap)
    loose_opt = PortfolioOptimizer(len(sleeve), loose_cfg.w_max, loose_cfg.solvers)
    log.info("Sensitivity sleeve: %d assets, long-only, cap %.0f%%, no turnover limit.",
             len(sleeve), 100 * loose_cfg.w_max)
    for lb in cfg.sensitivity_lookbacks:
        if lb >= len(returns) - 2 * TRADING_DAYS:
            log.warning("Skipping lookback %d: not enough data.", lb)
            continue
        res = run_walk_forward(returns[sleeve], rf_annual, rebal_pos, loose_cfg, loose_opt, lookback=lb,
                               objectives=("min_variance",), include_equal_weight=False)
        for r in res.values():
            m = performance_metrics(r, rf_daily)
            rows.append({"lookback": lb, "T/N": lb / len(sleeve), "estimator": r.estimator,
                         "realized_vol": m["Volatility"], "sharpe": m["Sharpe"],
                         "turnover": m["Annual turnover"], "risk_ratio": m["Risk forecast ratio"],
                         "eff_n": m["Effective N"], "max_weight": m["Max weight"],
                         "start": r.returns.index[0]})
    return pd.DataFrame(rows)


def fig_sensitivity(df: pd.DataFrame) -> go.Figure:
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
                        subplot_titles=("Realized volatility, min variance", "Risk forecast ratio"))
    for est in CFG.estimators:
        sub = df[df["estimator"] == est]
        for col, metric, fmt in ((1, "realized_vol", ".1%"), (2, "risk_ratio", ".2f")):
            fig.add_trace(go.Scatter(
                x=sub["T/N"], y=sub[metric], mode="lines+markers", name=LABELS[est], legendgroup=est,
                showlegend=(col == 1), line=dict(color=COLORS[est], width=2),
                marker=dict(size=9, line=dict(color=SURFACE, width=2)),
                customdata=sub["lookback"],
                hovertemplate=f"{LABELS[est]}<br>T/N %{{x:.1f}} (T=%{{customdata}})<br>%{{y:{fmt.replace('1%', '2%')}}}<extra></extra>"),
                row=1, col=col)
    fig.update_xaxes(title="T / N (observations per asset)", type="log",
                     tickvals=sorted(df["T/N"].unique()), ticktext=[f"{v:.1f}" for v in sorted(df["T/N"].unique())])
    fig.update_yaxes(tickformat=".2%", row=1, col=1)
    fig.add_hline(y=1.0, line=dict(color=INK_3, dash="dot"), row=1, col=2)
    fig.update_layout(title="Estimation error vs sample size · equity sleeve, long-only, no cap", height=500)
    _style_subplot_titles(fig)
    return _add_note(fig, "Each lookback starts trading on a different date, so compare estimators within a "
                          "lookback, not levels across lookbacks.")


SENSITIVITY = pd.DataFrame()
if CFG.run_sensitivity:
    SENSITIVITY = lookback_sensitivity(CFG)
    _register("sensitivity", fig_sensitivity(SENSITIVITY))
SENSITIVITY

## Cell 15 · Export dashboard and results
Writes a self-contained HTML dashboard (plotly.js embedded, works offline) plus CSVs for the README and further analysis.
Static PNGs are exported if `kaleido` is installed (`%pip install -q kaleido`).

In [ ]:
def export_dashboard(figs: Dict[str, go.Figure], metrics_display: pd.DataFrame, path: Path) -> Path:
    best = METRICS.drop(index=[i for i in METRICS.index if "buy & hold" in i], errors="ignore")
    best_name = best["Sharpe"].astype(float).idxmax()
    sections = []
    for i, (key, fig) in enumerate(figs.items()):
        html = fig.to_html(full_html=False, include_plotlyjs=True if i == 0 else False,
                           config={"displaylogo": False, "responsive": True})
        sections.append(f'<section class="card" id="{key}">{html}</section>')
    page = f"""<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Shrinkage frontier report</title>
<style>
 body{{margin:0;background:{SURFACE};color:{INK};font-family:Inter,Helvetica Neue,Arial,sans-serif}}
 main{{max-width:1180px;margin:0 auto;padding:32px 16px}}
 h1{{font-size:28px;margin:0 0 6px}} p.lede{{color:{INK_2};margin:0 0 24px;max-width:820px;line-height:1.5}}
 .kpis{{display:grid;grid-template-columns:repeat(auto-fit,minmax(200px,1fr));gap:12px;margin-bottom:24px}}
 .kpi{{border:1px solid {GRID};border-radius:8px;padding:14px 16px;background:white}}
 .kpi .v{{font-size:24px;font-weight:600}} .kpi .l{{color:{INK_2};font-size:13px}}
 .card{{border:1px solid {GRID};border-radius:8px;background:white;margin-bottom:20px;padding:8px;overflow-x:auto}}
 .card > div{{min-width:760px}}
 footer{{color:{INK_3};font-size:12px;margin-top:24px}}
</style></head><body><main>
<h1>Covariance shrinkage and the efficient frontier</h1>
<p class="lede">Walk-forward test of long-only min-variance and max-Sharpe portfolios on {N_ASSETS} ETFs,
{returns.index[rebal_pos[0]]:%b %Y} to {returns.index[-1]:%b %Y}. Every strategy uses the same expected
returns and constraints; only the covariance estimator changes. {_source_note}</p>
<div class="kpis">
 <div class="kpi"><div class="l">Best out-of-sample Sharpe</div><div class="v">{float(best.loc[best_name, 'Sharpe']):.2f}</div><div class="l">{best_name}</div></div>
 <div class="kpi"><div class="l">Median κ (correlation), sample</div><div class="v">{np.nanmedian(METRICS['Condition number'][METRICS['Estimator']=='Sample'].astype(float)):,.0f}</div></div>
 <div class="kpi"><div class="l">Median κ (correlation), Ledoit-Wolf</div><div class="v">{np.nanmedian(METRICS['Condition number'][METRICS['Estimator']=='Ledoit-Wolf'].astype(float)):,.0f}</div></div>
 <div class="kpi"><div class="l">Rebalances</div><div class="v">{len(rebal_pos)}</div><div class="l">monthly, {CFG.tc_bps:.0f} bps costs</div></div>
</div>
{''.join(sections)}
<footer>Generated {pd.Timestamp.now():%Y-%m-%d %H:%M}. Research code, not investment advice.</footer>
</main></body></html>"""
    path.write_text(page, encoding="utf-8")
    return path


def export_results(cfg: Config) -> None:
    out = cfg.output_dir
    METRICS.to_csv(out / "metrics.csv")
    METRICS_DISPLAY.to_csv(out / "metrics_formatted.csv")
    quality_report.to_csv(out / "data_quality_report.csv")
    pd.DataFrame({k: r.returns for k, r in RESULTS.items()}).to_csv(out / "daily_returns.csv")
    for r in RESULTS.values():
        if r.estimator != "equal_weight":
            slug = f"{r.estimator}_{r.objective}"
            r.weights.to_csv(out / f"weights_{slug}.csv")
            r.diagnostics.to_csv(out / f"diagnostics_{slug}.csv")
    if not SENSITIVITY.empty:
        SENSITIVITY.to_csv(out / "lookback_sensitivity.csv", index=False)
    try:
        import kaleido  # noqa: F401

        for key, fig in FIGURES.items():
            fig.write_image(out / "figures" / f"{key}.png", width=1200, height=fig.layout.height or 600, scale=2)
        log.info("PNG figures written to %s", out / "figures")
    except Exception as err:  # kaleido missing or no Chrome available
        log.info("Skipping PNG export (%s). HTML dashboard still has every chart.", type(err).__name__)


report_path = export_dashboard(FIGURES, METRICS_DISPLAY, CFG.output_dir / "report.html")
export_results(CFG)
log.info("Dashboard: %s", report_path)
print(METRICS_DISPLAY[["CAGR", "Volatility", "Sharpe", "Max drawdown", "Annual turnover",
                       "Risk forecast ratio", "JK p vs sample"]].to_string())